In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Agente LLM: comprar, vender ou manter (oito ativos)

A cada **cinco pregões**, uma LLM recebe apenas retornos e volatilidade conhecidos, manchetes verificadas já disponíveis e as posições que ela própria manteve. Ela decide `BUY`, `SELL` ou `HOLD` para cada ativo. Compra só entra em ativo ainda não mantido; venda só sai de ativo mantido. Os ativos mantidos têm pesos iguais; se nenhum estiver mantido, a carteira fica em caixa. Não há venda a descoberto, alavancagem, embeddings nem outra rede de previsão. Respostas JSON e contexto são salvos no Drive por provedor/modelo, com validação causal na retomada.

O teste 2022–23 já foi examinado: **qualquer desempenho é exploratório**. Nenhuma manchete confiável de 2024–25 foi acrescentada. Modelos treinados depois de 2023 podem memorizar resultados históricos. Os retornos são simulados da sessão **seguinte** à decisão após o fechamento; custos de 10 bps por nocional negociado. Compare no mesmo calendário com uma regra de compra/venda baseada nas previsões numéricas salvas, com caixa e com comprar e manter dos oito ativos. Os perfis não são recomendação financeira.

In [ ]:
import subprocess
import hashlib
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
DATA = Path('/content/drive/MyDrive/latent_fusion_data')
PANEL, NEWS = DATA / 'fusion_event_panel.npz', DATA / 'fusion_events.jsonl.gz'
BASELINE = DATA / 'fast_fusion_demo_results' / 'predictions_ts_only.csv'
for file in (PANEL, NEWS, BASELINE):
    if not file.is_file():
        raise FileNotFoundError(f'Falta artefato auditado no Drive: {file}')
{'painel': str(PANEL), 'manchetes': str(NEWS), 'previsões_numéricas': str(BASELINE)}

## Primeiro, referências reais sem chamar LLM

A regra numérica não é um resultado da LLM: entra quando sua previsão salva é positiva, sai quando negativa, e mantém nos demais casos. O resultado abaixo é apenas a referência histórica; o agente LLM não tem retorno medido até terminar a célula final. A previsão numérica é conferida contra cada ativo, data, rótulo e cobertura antes de qualquer chamada ao modelo.

In [ ]:
from IPython.display import display
from src.fusion.llm_trading_agent import preview_benchmarks
reference = preview_benchmarks(PANEL, NEWS, BASELINE, cost_bps=10)
display(reference[['return_pct', 'bh_return_pct', 'excess_pct', 'alpha_annual_pct', 'beta', 'turnover_total']].round(5))

## Selecionar provedor remoto

Defina no ambiente do Colab `LLM_CONTEXT_BACKEND` como `api` para um provedor de chat completions compatível com OpenAI, ou `transformers` para inferência 4-bit de um modelo *instruct* na GPU remota. Defina `LLM_CONTEXT_MODEL_ID` com o identificador exato; `LLM_CONTEXT_BASE_URL` é opcional para API HTTPS compatível. O identificador `gpt-6-luna` foi confirmado e utilizado pela API OpenAI; outros provedores precisam oferecer o mesmo protocolo. Para o modelo aberto, `Qwen/Qwen2.5-7B-Instruct` é um exemplo de identificador, sem garantia de caber na GPU específica. Não coloque chaves no notebook compartilhado.

Exemplo, em célula privada anterior: `os.environ['LLM_CONTEXT_BACKEND'] = 'transformers'`; `os.environ['LLM_CONTEXT_MODEL_ID'] = 'Qwen/Qwen2.5-7B-Instruct'`.

In [ ]:
import getpass
BACKEND = os.environ.get('LLM_CONTEXT_BACKEND')
MODEL_ID = os.environ.get('LLM_CONTEXT_MODEL_ID')
BASE_URL = os.environ.get('LLM_CONTEXT_BASE_URL') or None
if BACKEND not in ('api', 'transformers') or not MODEL_ID:
    raise ValueError('Defina LLM_CONTEXT_BACKEND e LLM_CONTEXT_MODEL_ID no Colab antes de prosseguir')
if BACKEND == 'api':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=2.0'], check=True)
    if not os.environ.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Chave da API (não será salva): ')
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Ative uma GPU remota para o modelo aberto')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44,<5', 'accelerate', 'bitsandbytes'], check=True)
OUT = DATA / 'llm_trading_agent_results' / (BACKEND + '-' + hashlib.sha256(f'{MODEL_ID}|{BASE_URL}'.encode()).hexdigest()[:12])
{'backend': BACKEND, 'modelo': MODEL_ID, 'saída': str(OUT)}

## Rodar o agente e medir resultados reais

O agente recebe **uma decisão por cesta a cada cinco sessões**, não 2.320 chamadas individuais; a quantidade exata de chamadas depende do calendário validado. Ação inválida, citação inventada ou cache incompatível interrompem a execução. Interrupções preservam as decisões válidas concluídas; após retomar, os prompts são refeitos com as posições que **a própria LLM escolheu**. Só ao completar todas as decisões são escritos métricas, posições, gráfico com BH e comparação de custos brutos/líquidos. O ledger da API guarda tokens e valores de tabela quando a resposta inclui `usage`; chamadas antigas sem esse campo ficam com custo desconhecido, não zero. Gastos de API em USD não são subtraídos dos retornos percentuais sem capital inicial informado. Resultados diferentes entre modelos não são evidência independente de alfa no teste já visto.

In [ ]:
from IPython.display import Image
from src.fusion.llm_trading_agent import run_agent
result = run_agent(PANEL, NEWS, BASELINE, OUT, backend=BACKEND, model_id=MODEL_ID,
                   base_url=BASE_URL, cost_bps=10)
display(result['metrics'][['return_pct', 'bh_return_pct', 'excess_pct', 'alpha_annual_pct', 'beta', 'turnover_total']].round(5))
display(result['actions'].head(16), Image(filename=str(OUT / 'agent_vs_benchmarks.png')))
display(result['cost_report']['trading'].round(5), result['cost_report']['api_summary'])
{'decisões_por_cesta': result['actions']['date'].nunique(), 'cache_auditável': str(result['cache']), 'métricas': str(OUT / 'portfolio_metrics.csv')}